In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("crude_ops.silver.drilling_enriched")

# Adding a time window (15 minutes) to see changes over time
gold_df = silver_df.withColumn(
    "GR_cleaned", 
    F.when(F.col("GR") <= 0, F.lit(None)).otherwise(F.col("GR"))
).groupBy(
    "phase", 
    "operation", 
    F.window("event_timestamp", "1 seconds") # to create multiple rows
).agg(
    F.avg("DEPTH").alias("avg_depth"),
    F.avg("GR_cleaned").alias("avg_gamma_ray"),
    F.count("*").alias("data_points_count")
).select(
    "window.start", 
    "phase", 
    "operation", 
    "avg_depth", 
    "avg_gamma_ray", 
    "data_points_count"
).orderBy("start")

gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("crude_ops.gold.drilling_stats_summary")

display(spark.table("crude_ops.gold.drilling_stats_summary"))

In [0]:
%skip
%sql
select * from crude_ops.gold.drilling_stats_summary

In [0]:
%skip
%sql
select distinct (event_timestamp)
from crude_ops.silver.drilling_enriched 
--where phase = 'drilling'


In [0]:
%skip
%sql
select * from crude_ops.bronze.drilling_raw